In [1]:
%load_ext autoreload
%reload_ext autoreload

import numpy as np
import time, logging, os

from genetic_algorithm.genetic_operators import mutate, crossover
from genetic_algorithm.schema import Network, RunState, HyperParameters
from genetic_algorithm.fitness import fitness_score, make_run_tag
from config.config import FITNESS_WEIGHTS, FILTER_OPTIONS

from utils.time_utils import format_elapsed, ElapsedTimeFormatter
from utils.checkpoint_utils import load_ga_checkpoint, save_ga_checkpoint, load_data

In [2]:
# Logger object. Use debug, info, warning, error accordingly
logger = logging.getLogger("genetic_algorithm")
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    handler = logging.StreamHandler()

    formatter = ElapsedTimeFormatter("[%(elapsed)s] %(levelname)s: %(message)s")

    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False

In [3]:
# GA VERSION 3 by sir Anthony and friends
from utils.graph_utils import load_graph


def nsga_ii(
    initial_population: list[Network],
    graph_of_stops,
    n_transport_nodes,
    hyperparameters: dict[str, any],
    city_name,
    run_tag="",
    checkpoint=None,
    debug=False
):
    # Sets logging to debug mode meaning logger.debug() works
    logger.setLevel(logging.DEBUG if debug else logging.INFO)
    hp = hyperparameters if isinstance(hyperparameters, HyperParameters) else HyperParameters.from_dict(hyperparameters)

    if checkpoint is None:
        state = RunState(network_population=initial_population)
        logger.info("No checkpoint found. Starting fresh GA run...")
    else:
        state = RunState.from_checkpoint(checkpoint)
        logger.info("Resumed at generation %d (%.1f s accumulated).", state.generation_num, state.elapsed_before)

    started = time.monotonic()

    def save():
        logger.info(f"Saving at generation {state.generation_num} ({elapsed()} s)...")
        save_ga_checkpoint(f"_{run_tag}", state.to_checkpoint(elapsed()), city_name)

    def elapsed() -> float:
        return state.elapsed_before + (time.monotonic() - started)
    
    graph_dir = os.path.join("../data/street_networks", f"{city_name}.graphml")

    allowed_edge = {}
    edge_len = {}
    total_network_road_distance = 0.0
    city_graph = load_graph(graph_dir)

    for u, v, k, data in city_graph.edges(keys=True, data=True):
        if data.get('junction') is not None:
            continue
        hw = data.get('highway')
        allowed = False
        if isinstance(hw, list):
            allowed = any(h in FILTER_OPTIONS for h in hw)
        else:
            allowed = hw in FILTER_OPTIONS

        key = frozenset((u, v))
        allowed_edge[key] = allowed
        edge_len[key] = data.get('length', 0.0)
        if isinstance(hw, str) and allowed:
            total_network_road_distance += edge_len[key]

    try:
        while state.generation_num <= hp.max_generations:
            generation_num = state.generation_num
            logger.debug(f"Generation {generation_num}")

            # Evaluate the fitness of each network in the population
            print(state)
            for network in state.network_population:
                if network.fitness_score is None:
                    network.fitness_score, network.fitness_score_components = fitness_score(
                        network, hp.fitness_weights, n_transport_nodes, total_network_road_distance, allowed_edge, edge_len
                    )

            # Sort the network population by fitness score
            sorted_network_population = sorted(state.network_population, key=lambda x: x.fitness_score, reverse=True)
            fitness_scores = np.array([network.fitness_score for network in sorted_network_population])

            if logger.isEnabledFor(logging.DEBUG):
                logger.debug("Gen %d scores: %s", generation_num, fitness_scores.tolist())

            max_score = sorted_network_population[0].fitness_score
            ave_score = np.average(fitness_scores)
            logger.info("Generation %d Max Score: %s Average Score: %s", generation_num, max_score, ave_score)

            # The max network score of this generation
            state.max_score_list.append([max_score, sorted_network_population[0].fitness_score_components])

            # Check if this generation has the highest max fitness score and highest average score
            if max_score >= state.best_max_score and ave_score >= state.best_avg_score:
                state.best_max_score, state.best_avg_score = max_score, ave_score
                state.best_population = sorted_network_population

            # Check if score plateaus based on max score
            if state.prev_max_score is not None and abs(max_score - state.prev_max_score) <= hp.plateau_threshold:
                state.plateau_counter = state.plateau_counter + 1
                if state.plateau_counter >= hp.plateau_count_limit:
                    logger.info(f"Plateau detected in generation {generation_num}. Stopping...")
                    break
            else:
                state.plateau_counter = 0
            state.prev_max_score = max_score

            # Choosing 10% of the networks to be parents using Roulette Wheel Selection
            logger.debug("Choosing parents...")

            # Getting the number of parents to be selected
            num_parents = int(len(state.network_population) * 0.10)
            num_parents = max(2, num_parents)
            if num_parents % 2 == 1:
                num_parents += 1

            logger.debug(f"Number of parents selected: {num_parents}")

            # Get the probabilities
            weights = fitness_scores - np.min(fitness_scores)
            total_score = np.sum(weights)
            if total_score > 0:
                selection_p = weights / total_score
                selection_p /= np.sum(selection_p)
            else:
                selection_p = np.full(len(weights), 1.0 / len(weights))

            parent_networks = np.random.choice(sorted_network_population, num_parents, replace=False, p=selection_p).tolist()

            # Pairing the parents
            parent_pairs = [(parent_networks[i], parent_networks[i+1]) for i in range(0, len(parent_networks), 2)]

            # Each parent pair will now produce children
            logger.debug("Generating children through crossover...")

            child_networks = []
            for (parent1, parent2) in parent_pairs:
                # Get 2 children from crossovers between the two parents
                child1, child2, is_different = crossover(parent1, parent2)

                # Getting the number of mutations. If the children are entirely different from their parents, there should be atleast one mutation
                table = hp.mutations_per_child if is_different else hp.mutations_per_child_identical
                num_mutations = int(np.random.choice(len(table), p=table))

                # Apply mutations to the children based on mutation probability hyperparameter
                for child in (child1, child2):
                    for _ in range(num_mutations):
                        if np.random.rand() < hp.mutation_probability:
                            mutate(child, graph_of_stops)

                # Add the children to the list of children
                child_networks.extend((child1, child2))

            # Remove the lowest scored networks and replace it with the children
            if child_networks:
                state.network_population = sorted_network_population[:-len(child_networks)] + child_networks

            state.generation_num += 1

            if state.generation_num % hp.checkpoint_every == 0:
                save()

            logger.debug(f"Completed generation {generation_num}. Proceeding to next generation...")

            # # Increment the generation number
            # state.generation_num += 1

    except KeyboardInterrupt:
        save()
        raise

    save()
    runtime = elapsed()
    logger.info(f"Highest score of all generations: {state.best_max_score}")
    logger.info(f"Total GA runtime: {format_elapsed(runtime)}")

    return state.best_population or [], state.max_score_list, runtime

In [4]:
HYPERPARAMETERS = dict(
  fitness_weights = FITNESS_WEIGHTS,
  mutation_probability = 0.2,
  mutations_per_child=(0.1, 0.1, 0.4, 0.2, 0.2),
  mutations_per_child_identical=(0.0, 0.1, 0.4, 0.3, 0.2),
  plateau_threshold=1e-11,
  plateau_count_limit=10_000,
  max_generations=1,
  checkpoint_every=10,
)

CITY_NAME = "Makati" # Taguig, Mandaluyong, Manila
run_tag = make_run_tag(HYPERPARAMETERS["fitness_weights"])

resume = False # Set to true if u wanna use a checkpoint
checkpoint = (load_ga_checkpoint(f"_{run_tag}", CITY_NAME) if resume else None)

INITIAL_RS_DIR = f"../data/initial_route_sets/{CITY_NAME}"

# file_path = os.path.join(INITIAL_RS_DIR, f"{CITY_NAME}_population.pkl")
# initial_population = load_data(file_path)

# file_path = os.path.join(INITIAL_RS_DIR, f"{CITY_NAME}_stops.pkl")
# graph_of_stops = load_data(file_path)

file_path = os.path.join(INITIAL_RS_DIR, f"{CITY_NAME}_population_w300_s20_l25_n50.pkl")
initial_population = load_data(file_path)

file_path = os.path.join(INITIAL_RS_DIR, f"{CITY_NAME}_stops_w300_s20_l25_n50.pkl")
graph_of_stops = load_data(file_path)

n_transport_nodes = sum(1 for _, d in graph_of_stops.nodes(data=True) if d.get("isTranspo"))

nsga_ii(initial_population, graph_of_stops, n_transport_nodes, HYPERPARAMETERS, CITY_NAME, run_tag, checkpoint, debug=False)

 2s] INFO: No checkpoint found. Starting fresh GA run...


Graph loaded successfully
RunState(network_population=[<genetic_algorithm.schema.Network object at 0x0000022B49171D30>, <genetic_algorithm.schema.Network object at 0x0000022B492220D0>, <genetic_algorithm.schema.Network object at 0x0000022B49223B10>, <genetic_algorithm.schema.Network object at 0x0000022B49200770>, <genetic_algorithm.schema.Network object at 0x0000022B492009D0>, <genetic_algorithm.schema.Network object at 0x0000022B49515910>, <genetic_algorithm.schema.Network object at 0x0000022B491A16A0>, <genetic_algorithm.schema.Network object at 0x0000022B491A18C0>, <genetic_algorithm.schema.Network object at 0x0000022B495DBB50>, <genetic_algorithm.schema.Network object at 0x0000022B49904450>, <genetic_algorithm.schema.Network object at 0x0000022B48E0E5D0>, <genetic_algorithm.schema.Network object at 0x0000022B49181B80>, <genetic_algorithm.schema.Network object at 0x0000022B499A1E10>, <genetic_algorithm.schema.Network object at 0x0000022B499A3150>, <genetic_algorithm.schema.Network o

 4s] INFO: Generation 1 Max Score: 0.16685230534450113 Average Score: 0.1106945518466961


KeyError: 133